# La familia de polinomios Go
### Exploración interactiva de un Hamiltoniano candidato

Este notebook permite explorar cualquier candidato del catálogo,
reproduce su análisis completo y genera visualizaciones.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import yaml, json

from src.families           import Hamiltonian, reference_hamiltonians
from src.algebra            import analyze
from src.topology           import compute_persistence, plot_persistence_diagram
from src.filter_candidates  import filter_candidate, robustness_test
from src.catalog            import Catalog

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)
print('Config cargada OK')

## 1. Selección del Hamiltoniano

In [ ]:
# ── Opción A: H_M1 de referencia ──────────────────────────────────────────────
h = reference_hamiltonians()[0]

# ── Opción B: candidato del catálogo ─────────────────────────────────────────
# cat = Catalog('../output/catalog.json')
# top = cat.top_n(1)[0]
# h   = Hamiltonian(top['template'], top['coefficients'])

# ── Opción C: definir manualmente ────────────────────────────────────────────
# h = Hamiltonian('cubic_mixed', {'a1':1,'a2':2,'b11':0,'b12':0,'b22':0,'c112':-1,'c122':-1})

print(h)

## 2. Análisis algebraico

In [ ]:
alg = analyze(h, cfg['analysis'])
print(f"Grado          : {alg['degree']}")
print(f"Soporte        : {alg['support']}")
print(f"Simetrías      : {alg['symmetries']}")
print(f"Nodos A₁       : {alg['n_nodes_A1']}")
print(f"Valores críticos: {[round(v,4) for v in alg['critical_values']]}")
print(f"Sep. mín. c*   : {alg['min_crit_separation']:.4f}")
print(f"\nH sobre pares Go:")
for pair, val in alg['values_on_valid_pairs'].items():
    print(f"  {pair:>12} → c = {val:.4f}")

## 3. Mapa de energía y fibras

In [ ]:
L  = cfg['analysis']['box_L']
N  = cfg['analysis']['grid_size']
xs = np.linspace(-L, L, N)
ys = np.linspace(-L, L, N)
X, Y = np.meshgrid(xs, ys)
Z = np.array(h(X, Y), dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(str(h), fontsize=10)

# Mapa de calor
im = axes[0].contourf(X, Y, Z, levels=30, cmap='RdBu_r')
plt.colorbar(im, ax=axes[0])
cvals = [p.get('c_val') for p in alg.get('critical_points',[]) if p.get('c_val')]
if cvals:
    axes[0].contour(X, Y, Z, levels=cvals, colors='orange',
                    linewidths=1.5, linestyles='--')
    cpts = alg['critical_points']
    axes[0].scatter([p['x'] for p in cpts], [p['y'] for p in cpts],
                    c='orange', s=80, marker='D', zorder=5,
                    label='nodo A₁')
axes[0].set_title('H(x,y) — mapa de energía'); axes[0].legend()
axes[0].set_xlabel('s₀'); axes[0].set_ylabel('s₁')

# Fibras H⁻¹(c) para c = valores críticos y juego
for c_lv, col, ls in [(cvals[0] if cvals else -1.24, 'orange', '--'),
                       (cvals[-1] if cvals else  1.24, 'orange', '--'),
                       (-1., '#4488FF', '-'), (1., '#FF8866', '-')]:
    axes[1].contour(X, Y, Z, levels=[c_lv], colors=[col],
                    linewidths=2, linestyles=ls)
axes[1].set_title('Fibras H⁻¹(c)'); axes[1].set_xlabel('s₀'); axes[1].set_ylabel('s₁')
axes[1].set_facecolor('#0A0A14')
plt.tight_layout(); plt.show()

## 4. Análisis TDA — Diagrama de persistencia

In [ ]:
tda = compute_persistence(h,
        L=L, N=N,
        tau=cfg['tda']['tau'],
        n_thresh=cfg['tda']['n_thresholds'])

print(f"H1 max lifetime  : {tda['max_h1_lifetime']:.4f}")
print(f"H1 barras largas : {tda['n_long_bars']}")
print(f"H1 vida media    : {tda['mean_h1_lifetime']:.4f}")
print(f"H0 barras        : {tda['h0_bars']}")
print(f"Profundidad pozo : {tda['well_depth']:.4f}")

fig, ax = plt.subplots(figsize=(5,5))
plot_persistence_diagram(tda, title='Diagrama de persistencia', ax=ax)
plt.tight_layout(); plt.show()

## 5. Filtrado y robustez

In [ ]:
filt = filter_candidate(alg, tda, cfg['filter'])
print(f"Pasa el filtro  : {filt['passes']}")
print(f"Criterios met   : {filt['criteria_met']}")
print(f"TDA score       : {filt['tda_score']:.4f}")

if filt['passes']:
    rob = robustness_test(h, filt['tda_score'],
                          {**cfg['analysis'], **cfg['filter'], **cfg['tda']})
    print(f"\nRobustez        : {rob['robustness']:.2%}")
    print(f"Score medio pert: {rob['mean_score_pert']:.4f} ± {rob['std_score_pert']:.4f}")

## 6. Catálogo — top candidatos

In [ ]:
try:
    cat = Catalog('../output/catalog.json')
    df  = cat.to_dataframe()
    print(f"Candidatos en catálogo: {len(df)}")
    print(df[df['passes']==True].sort_values('total_score', ascending=False)
           [['id','template','max_h1','n_nodes_A1','robustness','total_score']]
           .head(10).to_string(index=False))
except FileNotFoundError:
    print('Catálogo no encontrado. Ejecuta pipeline.py primero.')